# RF 기반 Post-training 시각화 실험

같은 사전학습 모델에 **RL·직접 역전파·DPO**를 적용해 피드백, 학습 신호, 벡터장, 생성 경로, 도착점 분포를 차례로 비교합니다.

**실행 장소: Google Colab.** 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 선택한 뒤 위에서 아래로 실행하세요. GPU가 없으면 Colab CPU를 사용합니다. 기본 실험은 시드 3개, 사전학습 2,500회, 추가 학습 100회입니다. 실행 시간은 할당된 장치에 따라 달라집니다.

- A: 매끄러운 보상에서 세 기법 비교, 기울기 접근 차단 조건 비교
- B: 성공·실패 보상, 목표 반경에 따른 학습 신호 비교
- C: 저장된 선호 기록의 범위 비교
- 그림의 축·범례는 폰트 설치 없이 표시되도록 영문이며 설명과 조작부는 한국어입니다.
- 이 노트북의 실행 결과는 저장소에 포함되어 있지 않습니다. 검증도 아래 Colab 셀에서 실행합니다.

## 1. 설정과 준비

첫 셀은 저장소 코드를 받아 Colab에 설치합니다. 기존 Colab PyTorch가 요구 범위에 있으면 유지합니다. 재현할 때는 `SOURCE_REF`를 원하는 커밋 SHA로 바꾸세요. 같은 런타임에서 소스 경로가 이미 존재하면 그 내용을 재사용하며 실제 커밋을 표시합니다.

In [ ]:
from pathlib import Path
import subprocess
import sys

try:
    from google.colab import output, files
except ImportError as error:
    raise RuntimeError("이 노트북은 Google Colab에서 실행하세요.") from error

REPOSITORY = "https://github.com/HisameOgasahara/tmp_fm_post.git"
SOURCE_REF = "main"
SOURCE_DIR = Path("/content/tmp_fm_post")
if not SOURCE_DIR.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(SOURCE_DIR)], check=True)
    subprocess.run(["git", "-C", str(SOURCE_DIR), "checkout", SOURCE_REF], check=True)
SOURCE_COMMIT = subprocess.check_output(
    ["git", "-C", str(SOURCE_DIR), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(SOURCE_DIR)], check=True)
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
output.enable_custom_widget_manager()
print("사용 소스 커밋:", SOURCE_COMMIT)

실험 설정은 다음 셀에서 관리합니다. `Config`의 나머지 기본값도 전체 설정 표에 표시됩니다. 설정을 바꾸어 다시 실험할 때는 이 셀부터 실행하면 새 결과 폴더를 만듭니다.

`RUN_SIGNAL_CHECKS`는 수치와 학습 신호의 일관성을 확인합니다. `RUN_INTEGRATION_CHECK`를 켜면 축소된 A·B·C부터 그림·애니메이션 내보내기까지 Colab에서 먼저 확인합니다. 축소 확인 결과는 실제 실험 결과와 섞이지 않습니다.

In [ ]:
from dataclasses import asdict
import json
import torch
import matplotlib.pyplot as plt
from IPython.display import display, HTML

from rf_lab import Config, Experiment
from rf_lab.checks import run_signal_checks, run_integration_check
from rf_lab.plots import (
    plot_data, plot_fields, plot_paths, plot_distributions,
    plot_one_update, plot_update_details, plot_learning_curves,
)
from rf_lab.widgets import show_one_update_controls, show_explorer
from rf_lab.export import export_all

CONFIG = Config(
    seeds=(0, 1, 2),
    device="auto",
    clusters=8,
    ring_radius=2.0,
    cluster_std=0.12,
    target_radius=0.35,
    reward_width=0.6,
    hidden_width=128,
    hidden_layers=3,
    steps=32,
    batch_size=256,
    eval_samples=2048,
    path_samples=12,
    grid_size=25,
    pretrain_updates=2500,
    pretrain_lr=1e-3,
    updates=100,
    checkpoint_every=10,
    learning_rates={"RL": 1e-4, "Direct": 1e-4, "DPO": 1e-4},
    regularization={"RL": 0.0, "Direct": 0.0, "DPO": 0.0},
    ppo_epochs=4,
    ppo_clip=0.2,
    sde_epsilon=0.02,
    sde_noise=0.3,
    dpo_beta=2.0,
    candidate_count=4096,
    preference_pairs=2048,
    coverage_exclusion_radius=0.7,
)
B_SMALL_RADIUS = 0.08
RUN_SIGNAL_CHECKS = True
RUN_INTEGRATION_CHECK = False
EXPORT_ANIMATIONS = True
DOWNLOAD_ZIP = True

CONFIG.validate()
print("장치:", "CUDA / " + torch.cuda.get_device_name(0)
      if CONFIG.device == "auto" and torch.cuda.is_available() else ("cpu" if CONFIG.device == "auto" else CONFIG.device))
print(json.dumps(asdict(CONFIG), ensure_ascii=False, indent=2))
if RUN_SIGNAL_CHECKS:
    run_signal_checks()
if RUN_INTEGRATION_CHECK:
    run_integration_check()

lab = Experiment(CONFIG)
(lab.output / "results" / "source_commit.txt").write_text(SOURCE_COMMIT, encoding="utf-8")

## 2. 데이터 확인

원 둘레의 8개 가우시안 군집과 오른쪽 위 목표 원을 확인합니다. 매끄러운 보상의 폭은 `reward_width`, 성공 여부의 경계는 `target_radius`가 정합니다. 두 설정은 서로 독립적입니다.

In [ ]:
fig = plot_data(CONFIG)
display(fig)
plt.close(fig)

## 3. RF 사전학습

작은 MLP가 좌표와 시간을 받아 속도를 예측합니다. 잡음과 데이터의 직선 보간에서 데이터 방향 속도를 학습합니다. 생성 시간은 **0에서 출발, 1에서 도착**입니다. 시드별 공통 사전학습 모델과 고정 평가 잡음을 저장합니다.

In [ ]:
lab.pretrain()

# 사전학습 상태만 그림 함수에 연결합니다. 추가 학습은 수행하지 않습니다.
preview = {
    "config": CONFIG,
    "reward_kind": "smooth",
    "records": {(seed, method): {0: lab.baselines[seed]}
                for seed in CONFIG.seeds for method in ("RL", "Direct", "DPO")},
}
lab.runs["_pretraining_preview"] = preview
try:
    fig = plot_fields(lab, "_pretraining_preview", step=0)
    display(fig)
    plt.close(fig)
    fig = plot_distributions(lab, "_pretraining_preview", step=0)
    display(fig)
    plt.close(fig)
finally:
    del lab.runs["_pretraining_preview"]

## 4. 공통 출발점 확인

모든 방법은 해당 시드의 사전학습 체크포인트를 복사합니다. 분포 평가는 고정 잡음 2,048개, 경로 그림은 그중 12개를 사용합니다. 공통 평가는 같은 Euler ODE 적분기로 수행합니다.

RL 수집은 내부 시간 구간의 SDE를 사용합니다. 양 끝은 짧은 ODE 단계로 연결하며 PPO에서 끝단 연결은 수집 당시 모델로 고정해 취급합니다. 내부 전이의 확률비를 최적화하는 근사 처리이며, 끝단까지 포함한 정확한 경로 우도 최적화로 해석하지 않습니다.

In [ ]:
for seed in CONFIG.seeds:
    checkpoint = lab.output / "checkpoints" / f"pretrained_seed{seed}.pt"
    print(f"시드 {seed}: {checkpoint.name} | 고정 평가 잡음 {tuple(lab.eval_noise[seed].shape)}")
print("사전학습 시간:", {seed: round(info["seconds"], 2) for seed, info in lab.pretraining.items()})

## 5. 한 번의 업데이트 확대

세 방법을 공통 사전학습 상태로 되돌려 한 번 업데이트합니다. RL 한 번은 새 경로 수집과 설정된 PPO epoch 수를 포함합니다. 왼쪽은 피드백, 가운데는 업데이트 전후 벡터장, 오른쪽은 같은 출발점에서 생긴 도착점 이동입니다.

직접 역전파의 **보상 기울기**와 **모델 업데이트 후 도착점 이동**은 서로 다른 패널에 표시합니다. 추가 그림은 RL의 같은 난수 SDE 경로, 단계별 역전파 신호, DPO 현재·기준 모델 오차를 보여줍니다. 샘플·시간·잡음·학습률은 결과 폴더에 저장됩니다.

In [ ]:
one_update = lab.compare_one_update()
for method, info in one_update["methods"].items():
    print(method, info["status"], "| 학습률:", info["learning_rate"])
for fig in (plot_one_update(lab, one_update), plot_update_details(one_update)):
    display(fig)
    plt.close(fig)

one_update_controls = show_one_update_controls(lab)

## 6. 실험 A — 매끄러운 보상

먼저 기울기 제공 조건을 실행하고, 같은 보상 값에서 기울기 연결만 차단한 조건을 실행합니다. 점수만 제공하면 직접 역전파는 업데이트를 건너뜁니다. RL과 DPO는 점수나 그 점수로 만든 선호 쌍을 사용합니다.

DPO는 현재 모델과 고정 기준 모델의 **flow-matching 예측 오차 차이**에 로지스틱 선호 손실을 적용합니다. 정확한 ODE 우도 기반 DPO와는 구별하는 실험용 대리 목적함수입니다. 쌍 안의 winner/loser는 같은 시간과 잡음을 사용합니다.

In [ ]:
lab.run("A_gradient", reward_kind="smooth", gradient_access=True)
lab.run("A_scores_only", reward_kind="smooth", gradient_access=False)
for function in (plot_fields, plot_paths, plot_distributions):
    fig = function(lab, "A_gradient")
    display(fig)
    plt.close(fig)
fig = plot_fields(lab, "A_gradient", delta=True)
display(fig)
plt.close(fig)
fig = plot_learning_curves(lab, ["A_gradient", "A_scores_only"])
display(fig)
plt.close(fig)

## 7. 실험 B — 성공 여부 보상

목표 안은 1, 밖은 0입니다. 직접 역전파는 사용할 방향 신호가 없어 건너뜁니다. RL은 배치 안의 성공·실패 차이가 필요하고, DPO는 성공·실패 선호 쌍이 필요합니다.

기본 반경과 작은 반경을 각각 같은 사전학습 상태에서 실행합니다. 작은 반경에서도 성공이 발생할 수 있으므로 실제 성공 샘플과 유효 쌍 수로 판단하세요. 위의 한 업데이트 조작부에서 반경을 더 바꾸어 볼 수 있습니다.

In [ ]:
lab.run("B_default", reward_kind="binary")
lab.run("B_small", reward_kind="binary", target_radius=B_SMALL_RADIUS)
for name in ("B_default", "B_small"):
    print("\n", name)
    final_rows = [row for row in lab.runs[name]["rows"] if row["step"] == CONFIG.updates]
    for row in final_rows:
        print(f"시드={row['seed']} {row['method']}: {row['status']} | "
              f"성공 샘플={row['success_samples']} | 유효 쌍={row['valid_pairs']} | "
              f"도달률={row['success_rate']:.1%}")
    fig = plot_distributions(lab, name)
    display(fig)
    plt.close(fig)

## 8. 실험 C — 고정된 선호 기록

다음 셀에서 사전학습 모델의 후보와 보상을 한 번 수집합니다. 넓은 기록은 전체 후보를 사용하며, 목표 주변 기록 부족 조건은 목표 중심에서 `coverage_exclusion_radius` 이내의 후보를 제외합니다. 두 조건 모두 같은 수의 유효 선호 쌍을 복원 추출합니다. 후보가 충분히 목표와 8개 군집을 덮는지는 아래 표시로 확인합니다.

In [ ]:
from rf_lab.core import cluster_centers, target_center
saved_records = lab.prepare_preferences("C_saved", reward_kind="smooth", coverage=True)

for seed, record in saved_records.items():
    candidates = record["source"]["candidates"]
    near = (candidates - target_center(CONFIG, candidates.device)).norm(dim=-1) <= CONFIG.coverage_exclusion_radius
    nearest = torch.cdist(candidates, cluster_centers(CONFIG, candidates.device)).argmin(dim=1)
    print(f"시드 {seed}: 후보 {len(candidates)}개, 목표 주변 {int(near.sum())}개, "
          f"최근접 군집별 후보 수 {torch.bincount(nearest, minlength=CONFIG.clusters).tolist()}")
    counts = {condition: 0 if pairs is None else len(pairs["winner"])
              for condition, pairs in record["datasets"].items()}
    print("저장한 선호 쌍:", counts)
    if any(count == 0 for count in counts.values()):
        raise RuntimeError("유효 선호 쌍이 부족합니다. 후보 수 또는 사전학습 설정을 조정한 뒤 새 실험을 시작하세요.")
    if not near.any() or len(nearest.unique()) != CONFIG.clusters:
        raise RuntimeError("넓은 기록이 목표 주변과 8개 군집을 덮지 못했습니다. 사전학습 또는 후보 수를 조정하세요.")
print("선호 기록 저장 완료:", lab.output / "results")

이제 **새 학습용 평가: 사용 불가 / 저장된 유효 선호 쌍: 사용 가능**인 조건입니다. 학습은 저장된 쌍만 사용합니다. 그래프용 보상 평가는 학습 입력과 분리하며 별도 비용으로 기록합니다.

In [ ]:
print("새 학습용 평가: 사용 불가")
print("저장된 유효 선호 쌍: 사용 가능")
lab.run("C_wide", preference_name="C_saved", coverage="wide")
lab.run("C_missing_target", preference_name="C_saved", coverage="missing")
fig = plot_learning_curves(lab, ["C_wide", "C_missing_target"])
display(fig)
plt.close(fig)
for name in ("C_wide", "C_missing_target"):
    fig = plot_distributions(lab, name)
    display(fig)
    plt.close(fig)

## 9. 결과 탐색

실험·시드·학습 진행 시점·생성 시점을 바꾸어 봅니다. 벡터장은 공통 좌표·격자·화살표 배율을 사용합니다. 변화량 그림은 패널 간 같은 색 범위를 사용하며, 원시 변화량과 단위 방향을 전환할 수 있습니다.

군집 밖 점은 가장 가까운 중심으로부터 군집 표준편차의 `outside_sigma`배보다 먼 점으로 정의합니다. 보상 상승과 함께 군집 밖 비율, 원래 군집별 비율을 확인하세요.

In [ ]:
explorer = show_explorer(lab)

## 10. 내보내기

실험별 그림·애니메이션, 시드별 지표 CSV, 설정, 사전학습 및 추가 학습 체크포인트, 한 업데이트의 상세 기록과 한국어 결과 보고서를 저장합니다. 최상위 대표 그림·애니메이션은 첫 번째 실험 A의 결과이며 하위 폴더에 모든 실험이 있습니다.

애니메이션은 Colab에서 생성하므로 시간이 더 걸릴 수 있습니다. `EXPORT_ANIMATIONS=False`이면 정적 그림만 저장합니다. ZIP에는 큰 체크포인트 파일도 포함됩니다. 모델 파일은 이 노트북이 생성한 파일만 사용하고, 읽을 때는 `torch.load(..., weights_only=True)`를 사용하세요.

In [ ]:
archive_path = export_all(lab, animate=EXPORT_ANIMATIONS)
print("결과 보고서:", lab.output / "실험결과.md")
if EXPORT_ANIMATIONS:
    display(HTML((lab.output / "animations" / "training_comparison.html").read_text(encoding="utf-8")))
if DOWNLOAD_ZIP:
    files.download(archive_path)

## 결과 읽는 순서

1. 실제로 사용 가능한 신호와 건너뛴 기법을 확인합니다.
2. 같은 위치·시점의 화살표가 어느 방향으로 얼마나 달라졌는지 살펴봅니다.
3. 같은 출발점의 경로와 도착점이 어떻게 달라졌는지 연결합니다.
4. 학습용 평가 수·시간·GPU 메모리와 함께 비교합니다.

얇은 곡선은 시드별 결과, 굵은 곡선은 같은 업데이트 횟수에서 비용과 성능을 각각 평균한 값입니다. DPO는 후보를 최초 평가한 뒤 평가 수가 늘지 않으므로 평가 수 축에서는 세로 선이 나타날 수 있습니다. 공통 사전학습과 평가·그림·파일 저장 비용은 학습 비용에서 분리됩니다.